# 訂正にフィードバックコメントを付ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_gec_education/feedback_comment.ipynb)

NLP クックブック 26-13 ／ 文法誤り訂正と言語教育

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

学習者に、誤りを直した結果だけでなく、「なぜ誤りなのか」「どう直せばよいか」の説明を返したい。

In [1]:
import difflib
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる (LLM の章の「オープンな LLM とチャットする」を参照)。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=160):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def diff(source, corrected):
    """訂正箇所を「元 → 訂正後」の組として取り出す"""
    ops = difflib.SequenceMatcher(None, source, corrected, autojunk=False).get_opcodes()
    return [(source[i1:i2] or "(なし)", corrected[j1:j2] or "(削除)") for op, i1, i2, j1, j2 in ops if op != "equal"]

pairs = [("私は毎日学校を行きます。", "私は毎日学校に行きます。"), ("この料理はとてもおいしいでした。", "この料理はとてもおいしかったです。"),
         ("日本に来たばかりので、日本語がまだ下手です。", "日本に来たばかりなので、日本語がまだ下手です。"), ("彼は私にプレゼントをもらいました。", "彼は私にプレゼントをくれました。")]
for s, c in pairs:
    print(s, "→", c, diff(s, c))

私は毎日学校を行きます。 → 私は毎日学校に行きます。 [('を', 'に')]
この料理はとてもおいしいでした。 → この料理はとてもおいしかったです。 [('い', 'かった'), ('した', 'す')]
日本に来たばかりので、日本語がまだ下手です。 → 日本に来たばかりなので、日本語がまだ下手です。 [('(なし)', 'な')]
彼は私にプレゼントをもらいました。 → 彼は私にプレゼントをくれました。 [('もらい', 'くれ')]


In [2]:
PROMPT = """あなたは日本語教師です。日本語学習者の文と、その訂正を示します。
訂正箇所について、学習者向けのフィードバックを書いてください。
- なぜ元の表現が誤りなのかを、文法の規則として 1〜2 文で説明する
- 専門用語はできるだけ使わず、やさしい日本語で書く
- 同じ規則を使った正しい例文を 1 つ挙げる
- あいさつや前置きは書かず、全体で 150 字以内にする

学習者の文: {source}
訂正後の文: {corrected}
訂正箇所: {edits}"""

for s, c in pairs:
    edits = "、".join(f"「{a}」→「{b}」" for a, b in diff(s, c))
    print(f"■ {s} → {c}\n{chat(PROMPT.format(source=s, corrected=c, edits=edits))}\n")

■ 私は毎日学校を行きます。 → 私は毎日学校に行きます。
「を」ではなく「に」を使うのは、移動の目的地を表すときのルールです。例えば、「家に帰ります」のように言います。あなたの文では、「学校に行く」という行動の目標地点を明確にするために「に」が必要です。正しい表現を使うことで、より自然な日本語になります。例えば、「毎朝、公園に散歩に行きます」というように言えます。

■ この料理はとてもおいしいでした。 → この料理はとてもおいしかったです。
「おいしい」は形容詞なので、過去形には「かった」を使います。また、「食べる」などの動詞の過去形を作る「た」と混同しないように、「す」ではなく「ます形」の過去形の「ました」を使うのが正解です。例えば、「このケーキはおいしいです」が正しい表現で、過去に食べた場合は「とてもおいしかったです」となります。

■ 日本に来たばかりので、日本語がまだ下手です。 → 日本に来たばかりなので、日本語がまだ下手です。
「日本に来たばかり」という表現は、ある出来事が最近起こったことを表すために使います。しかし、「来たばかり」だけでは、現在どのような状態かを説明するのに十分ではありません。そこで、「ので」という接続詞を使って、その理由を説明します。つまり、「日本に来たばかりなので、~です」という形で、最近日本に来たという事実が現在の状態に影響を与えていることを表します。

例えば、「私は先月引っ越してきたばかりなので、まだ新しい環境に慣れていないのです。」このように、「ので」を使うことで、最近の状況と現在の状態が結びつくことを明示できます。

元の表現「日本に来たばかりので」は、文法的に誤りです。正しくは「日本に来たばかりなので」とする必要があります。この修正により、学習者がどのように recentness（最近であること）と current state（現在の状態）を結びつけるべきかが明確になります。

■ 彼は私にプレゼントをもらいました。 → 彼は私にプレゼントをくれました。
元の表現「彼は私にプレゼントをもらいました」は少し不自然です。なぜなら、「もらう」は一般的に受身の意味を持ち、この場合「彼が私に対して何かをする」という動作主を明確に示すためには「くれる」を使うのが自然だからです。例えば、「彼は私に本をくれました」のように言うと、より

In [3]:
# 訂正と説明を別々に行う理由: LLM に一度に任せると、説明が訂正と食い違うことがある。説明に訂正後の表現が含まれているかを機械的に確かめる
def check(source, corrected, comment):
    return {"訂正後の表現に触れている": all(b in comment for a, b in diff(source, corrected) if b != "(削除)"),
            "長すぎない": len(comment) <= 300}

s, c = pairs[0]
comment = chat(PROMPT.format(source=s, corrected=c, edits="、".join(f"「{a}」→「{b}」" for a, b in diff(s, c))))
print(check(s, c, comment))

{'訂正後の表現に触れている': True, '長すぎない': True}


- フィードバックコメント生成は、訂正 (何を直すか) とは別のタスクとして研究されている。学習効果のためには、答えを示すだけでなく、規則に気付かせる説明が重要とされる。
- 訂正箇所を差分で明示してから説明させると、どの誤りについての説明かがぶれにくい。誤りが複数あるときは、1 か所ずつ説明させる。
- 長さの指示は守られにくい。この実験でも、3 つ目の説明は「150 字以内」の指示を大きく超えて 400 字近くになり、「recentness」のような英語まで混ざった。1 つ目も 150 字をわずかに超えている (下の点検は上限を 300 字にしているので通る)。生成後に長さを点検し、超えたら要約させ直す、といった後処理を用意する。
- 説明が訂正と食い違うこともある。2 つ目の説明は、訂正後の文 (「おいしかったです」) に反して「「ました」を使うのが正解」と書いている。
- 小さな LLM は、もっともらしいが誤った文法説明を書くことがある (存在しない規則を作る、助詞の働きを取り違える)。学習者にそのまま見せる前に、教師による確認か、誤りの種類ごとに用意した説明文 (テンプレート) との併用を検討する。
- 誤りの種類が分かっていれば (助詞の選択、時制、活用など)、種類ごとに人が書いた説明を引いてくる方法が最も安全。誤りタイプの判定には、この章の ERRANT のレシピの考え方が使える。
- 学習者のレベルに合わせて説明の言葉を変える (初級者には母語で、など) ことも、LLM への指示で行える。